# arXiv 初筛调试
选择 **conda ml / Python (ml)** 内核，按顺序运行。这里只请求公开 arXiv 元数据并运行本地分类/关键词规则：不调用 LLM、不下载正文、不读取或修改正式状态、不推送。

默认检查北京时间**昨天结束的前 14 个完整自然日**，按论文首次提交时间筛选，不是修订时间。抓取完整分类元数据后缓存，再在本地反复修改规则。缓存结果中的论文版本是抓取时 API 返回的版本。

In [ ]:
from pathlib import Path
import json
import sys

if Path(sys.executable).parent.name.lower() != 'ml':
    raise RuntimeError(f'请先切换到 conda ml 内核。当前 Python：{sys.executable}')

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'config.toml').is_file() and (p / 'app/arxivdaily').is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('D:/CodexProject/ArxivDaily')  # 从项目外启动时，可修改这里
if not (PROJECT_ROOT / 'config.toml').is_file():
    raise RuntimeError('找不到项目，请修改 PROJECT_ROOT。')
sys.path[:0] = [str(PROJECT_ROOT / 'app/.runtime'), str(PROJECT_ROOT / 'app')]

from IPython.display import HTML, display
from arxivdaily.config import load_config
from tools.debug_rules import (date_window, load_or_fetch, screen_snapshot,
                               route_counts, compare_matches, inspect_paper,
                               table_html, export_matches)

CONFIG_PATH = PROJECT_ROOT / 'config.toml'
CACHE_DIR = PROJECT_ROOT / 'app/tmp/rules-debug'
print('Python:', sys.executable)
print('配置:', CONFIG_PATH)


## 1. 选择时间与分类
改 DAYS、END_DATE；分类读取根 config.toml 的 `[rules].categories`。END_DATE 为 None 时使用昨天，也可以填 `'2026-10-06'`。
MAX_METADATA 是本次调试的原始论文数上限，不改正式配置；超过上限或网络出错时会显示“不完整”，不能视为该范围的全部结果。公开 API 分页较慢，默认每 100 条请求间隔至少 3 秒。

In [ ]:
DAYS = 14
END_DATE = None               # 例如 '2026-10-06'；None = 北京时间昨天
TIMEZONE = 'Asia/Shanghai'
REFRESH = True               # 缓存存在时复用；True = 重新请求 arXiv
MAX_METADATA = 10000          # 调试抓取上限，不改正式 max_candidates
DISPLAY_LIMIT = 100           # 只限制表格展示数量，完整结果在 matches

config = load_config(CONFIG_PATH)
window = date_window(DAYS, END_DATE, timezone_name=TIMEZONE)
print('日期（首尾都包含）:', window['local_start_date'], '至', window['local_end_date'])
print('分类:', config['rules']['categories'] or '所有分类')
print('实际 UTC 查询:', window['start'], '至', window['end'])


## 2. 抓取或复用元数据
**只有这个单元会联网。** 缓存保留所选分类/日期内的全部已抓到元数据，包括没有命中当前关键词的论文，避免扩大关键词时漏掉旧缓存中的负例。
数据保存到忽略目录 app/tmp/rules-debug；首次抓取可能需要数分钟。缓存不会自动检查远端是否出现新数据；要更新它将 REFRESH 设 True 后重新运行参数单元和本单元。

In [ ]:
snapshot, cache_path, reused = load_or_fetch(
    config, window, CACHE_DIR, refresh=REFRESH, max_papers=MAX_METADATA
)
print('来源:', '本地缓存' if reused else '本次 arXiv 请求')
print('抓取时间 UTC:', snapshot['fetched_at'])
print('元数据篇数:', len(snapshot['papers']))
print('完整性:', '完整' if snapshot['complete'] else '不完整：以下只是目前已抓到的部分')
print('缓存:', cache_path)
if snapshot['errors']:
    print('覆盖问题:', json.dumps(snapshot['errors'], ensure_ascii=False, indent=2))


## 3. 重新读取配置并运行初筛
改完 `[rules]`、`[rules.groups]`、`[rules.routes]` 后保存 config.toml，再运行**这个单元**。每次都会重新读文件；不联网、不参考正式已推送记录。
路线为 OR，组内同义词为 OR，每条路线的内层列表为 AND。一篇可命中多路线，因此路线计数之和可以大于论文总数。禁用默认路线设 `[]`，不要只删除那一行。

In [ ]:
config = load_config(CONFIG_PATH)
previous_matches = globals().get('matches')
new_matches = screen_snapshot(snapshot, config, window=window)
comparison = compare_matches(previous_matches, new_matches) if previous_matches is not None else None
matches = new_matches
print('抓取覆盖:', '完整' if snapshot['complete'] else '不完整，不代表全部结果')
print('候选元数据:', len(snapshot['papers']), '；初筛命中:', len(matches))
print('各路线命中:', route_counts(matches))
if comparison is not None:
    print('与上次初筛相比:', {name: len(rows) for name, rows in comparison.items()})
display(HTML(table_html(matches, DISPLAY_LIMIT)))


## 4. 看一篇为什么命中或未命中
把 PAPER_ID 改成缓存中任一论文的基础 ID 或版本 ID。可以检查未入选者，查看英文摘要、分类是否通过、排除词、具体命中的同义词与路线。没有分类/日期覆盖的论文需要先调整范围重新抓取。

In [ ]:
PAPER_ID = matches[0]['base_id'] if matches else None  # 可替换为 '2610.12345'
if PAPER_ID:
    detail = inspect_paper(snapshot, config, PAPER_ID)
    print(json.dumps(detail, ensure_ascii=False, indent=2))
else:
    print('本次没有初筛命中；可从 snapshot["papers"] 中选 ID 检查。')


## 5. 查看规则调整前后的差异
修改配置再运行第 3 步后，这里显示新增、移除和命中路线改变的论文。对比依赖上一次的 matches，不会读正式状态。对比不同日期或分类范围时，差异不只来自关键词变化，应结合范围理解。

In [ ]:
if comparison is None:
    print('已有首次结果。修改配置后重跑第 3 步，再查看差异。')
else:
    for name, label in [('added', '新增'), ('removed', '移除'), ('changed_routes', '路线改变')]:
        print(label, len(comparison[name]), '篇')
        display(HTML(table_html(comparison[name], DISPLAY_LIMIT)))


## 6. 可选：导出完整结果
CSV 包含英文摘要与命中依据，保存在本地临时目录。导出不联网，也不会写正式论文索引。

In [ ]:
export_path = export_matches(matches, cache_path.with_name(cache_path.stem + '-matches.csv'))
print('已导出:', export_path)


## 常用调试顺序
- 改关键词/组合/排除词/标题开关：保存根配置 → 重跑第 3 步 → 检查第 4、5 步。
- 改日期或分类：保存配置 → 重跑参数单元 → 第 2 步抓取 → 第 3 步筛选。扩大分类而只重跑筛选会明确报错；缩小分类可在旧缓存上筛选。
- 补齐网络失败/达到上限后的覆盖：查看 snapshot['errors'] 与 snapshot['shards']；可缩小日期或分类，或在 10000 上限内调大 MAX_METADATA 后重新抓取。
- 更新缓存里的远端新数据：REFRESH=True → 重跑参数及抓取单元；完成后设回 False。
- 修改提示与模型参数不会影响本 notebook，因为这里止于规则初筛。真实模型质量需另做少量隔离 replay。

若没有 ml 内核，在 conda ml 中运行 `python -m ipykernel install --user --name ml --display-name "Python (ml)"` 注册；如果该环境没有 ipykernel，请先在 ml 中安装它。本 notebook 本身不安装或修改环境。